# GPROF V8 GMI Stats

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

In [ ]:
from gprof_nn.metrics import MeanPrecip


In [ ]:
results_gmi_v7 = sorted(list(Path("/pdata4/archive/GPM/2A-CLIM_GMI_V7/").glob("**/*.HDF5")))
stats_gmi_v7 = MeanPrecip()
stats_gmi_v7.compute(results_gmi_v7, n_processes=8)
len(results_gmi_v7)

In [ ]:
results = stats_gmi_v7.get_results()

In [ ]:
plt.pcolormesh(results.global_mean)

In [ ]:
results_amsr2 = sorted(list(Path("/edata2/simon/gprof_v8/results/amsr2/gprof_nn_3d/").glob("*.202001*.nc")))
stats_amsr2 = MeanPrecip()
stats_amsr2.compute(results_amsr2, n_processes=8)

In [ ]:
results_atms = sorted(list(Path("/edata2/simon/gprof_v8/results/atms/gprof_nn_3d/").glob("*.202001*.nc")))
stats_atms = MeanPrecip()
stats_atms.compute(results_atms, n_processes=8)

In [ ]:
results_gmi = sorted(list(Path("/edata2/simon/gprof_v8/results/gmi/gprof_nn_3d/").glob("*.nc")))
stats_gmi = MeanPrecip()
stats_gmi.compute(results_gmi, n_processes=8)

In [ ]:
results_gmi_v7 = stats_gmi_v7.get_results()
results_gmi = stats_gmi.get_results()
results_amsr2 = stats_amsr2.get_results()

In [ ]:
plt.plot(results_gmi.latitude, results_gmi.zonal_mean, c="C0")
plt.plot(results_gmi.latitude, results_gmi_v7.zonal_mean, c="C0", ls="--")
plt.plot(results_gmi.latitude, results_amsr2.zonal_mean, c="C1")

In [ ]:
plt.pcolormesh(100.0 * (results_gmi.global_mean - results_gmi_v7.global_mean) / results_gmi.global_mean, vmin=-50, vmax=50, cmap="coolwarm")
plt.colorbar()

In [ ]:
plt.pcolormesh(results_gmi_v7.global_mean)
plt.colorbar()

In [ ]:
plt.pcolormesh(results_gmi.global_mean)

In [ ]:
from matplotlib.colors import LogNorm
plt.pcolormesh(results_gmi.global_mean / results_gmi_v7.global_mean, norm=LogNorm())
plt.colorbar()

## Data

We use collocation between ATMS and GPM CMB and match them with the corresponding GPROF simulator files.


In [ ]:
collocations = sorted(list(Path("/data/gprof_v8/collocations/mrms/gmi/gridded/").glob("*2020082*.nc")))
collocations = np.random.permutation(collocations)[:2_000]

In [ ]:
len(collocations)

In [ ]:
def load_gprof_nn_results(base_path: Path, collocation_file: Path) -> Path:
    """
    Load GPROF-NN results for given collocation files.

    Args:
        base_path: A path object pointing to to the directory containing the GPROF-NN retrieval results.
        collocation_file: A path object pointing to the collocation files.

    Return:
        The retrieval results as xr.Dataset.
    """
    base_path = Path(base_path)
    return xr.load_dataset(base_path / collocation_file.name)

In [ ]:
from datetime import datetime
from pansat.products.satellite.gpm import l2a_gprof_gpm_gmi_v07a
import hdf5plugin

def load_gprof_results(collocation_file: Path) -> xr.Dataset:
    """
    Load GPROF V7 results.

    Args:
        collocation_file: A path object pointing to the retrieval results.

    Return:
         The retrieval results as a xarray.Dataset
    """
    date = datetime.strptime(collocation_file.name.split("_")[-1], "%Y%m%d%H%M%S.nc")
    with xr.open_dataset(collocation_file, group="input_data") as data:
        scan_start = data.attrs["scan_start"]
        scan_end = data.attrs["scan_end"]
        gprof_recs = l2a_gprof_gpm_gmi_v07a.get(date)
        gprof_data = l2a_gprof_gpm_gmi_v07a.open(gprof_recs[0])[{"scans": slice(scan_start, scan_end)}]
    gprof_data = gprof_data.rename(surface_precipitation="surface_precip").drop_vars(("latitude", "longitude"))
    sp = gprof_data.surface_precip.data
    sp[sp < 0] = np.nan
    return gprof_data

In [ ]:
gprof_nn_3d_path = Path("/data/gprof_v8/results/mrms/gmi/gprof_nn_3d/")
gprof_nn_1d_path = Path("/data/gprof_v8/results/mrms/gmi/gprof_nn_1d/")

In [ ]:
from gprof_nn.metrics import Evaluator

In [ ]:
from functools import partial
gprof_evaluator = Evaluator(load_gprof_results, collocations)
gprof_nn_3d_evaluator = Evaluator(partial(load_gprof_nn_results, gprof_nn_3d_path), collocations)
gprof_nn_1d_evaluator = Evaluator(partial(load_gprof_nn_results, gprof_nn_1d_path), collocations)

In [ ]:
gprof_evaluator.plot_results(49)

In [ ]:
gprof_nn_3d_evaluator.plot_results(49)

In [ ]:
gprof_nn_1d_evaluator.plot_results(49)

In [ ]:
gprof_nn_1d_evaluator.collocation_files[49]

In [ ]:
results_gprof = gprof_evaluator.get_retrieval_results(49)
results_gprof_nn_1d = gprof_nn_1d_evaluator.get_retrieval_results(49)
results_gprof_nn_3d = gprof_nn_3d_evaluator.get_retrieval_results(49)
results_mrms = gprof_nn_3d_evaluator.get_reference_results(49)

In [ ]:
gprof_evaluator.collocation_files[49]

## Case study

In [ ]:
lon_min = -97
lon_max = -87
lat_min = 27
lat_max = 37

lons = results_mrms.longitude.data
lats = results_mrms.latitude.data
lon_mask = (lon_min <= lons) * (lons <= lon_max)
lat_mask = (lat_min <= lats) * (lats <= lat_max)
## Case study
results_gprof = results_gprof[{"longitude": lon_mask, "latitude": lat_mask}]
results_mrms = results_mrms[{"longitude": lon_mask, "latitude": lat_mask}]
results_gprof_nn_1d = results_gprof_nn_1d[{"longitude": lon_mask, "latitude": lat_mask}]
results_gprof_nn_3d = results_gprof_nn_3d[{"longitude": lon_mask, "latitude": lat_mask}]

In [ ]:
swath = SwathDefinition(*np.meshgrid(lons[lon_mask], lats[lat_mask]))
background = get_blue_marble(swath)

In [ ]:
import cartopy.crs as ccrs
from gprof_nn.plotting import add_ticks, scale_bar, get_blue_marble
from matplotlib.gridspec import GridSpec
from matplotlib.colors import Normalize, LogNorm
from pyresample.geometry import SwathDefinition

fig = plt.figure(figsize=(26, 6))
gs = GridSpec(1, 5, width_ratios=[1.0, 1.0, 1.0, 1.0, 0.075])
crs = ccrs.PlateCarree()

levels = np.logspace(-1, 2, 11)
norm = LogNorm(1e-1, 1e1)
levels = np.linspace(0, 20, 11)
norm = Normalize(0, 20)

cmap = "flare"

lon_ticks = [-96, -93, -90]
lat_ticks = [28, 31, 34]

ax = fig.add_subplot(gs[0, 0], projection=crs)
lons = results_mrms.longitude.data
lats = results_mrms.latitude.data
ax.pcolormesh(lons, lats, background)
sp_mrms = results_mrms.surface_precip.data
sp_mrms[sp_mrms < 0] = np.nan
sp_mrms = np.maximum(sp_mrms, 1e-3)
ax.pcolormesh(lons, lats, background)
ax.contourf(lons, lats, sp_mrms, norm=norm, cmap=cmap, levels=levels, extend="both")
ax.coastlines(color="grey")
ax.set_title("(a) MRMS", loc="left")
scale_bar(ax, 400e3, textcolor="w", border_color="w")
add_ticks(ax, lons=lon_ticks, lats=lat_ticks)

ax = fig.add_subplot(gs[0, 1], projection=crs)
lons = results_mrms.longitude.data
lats = results_mrms.latitude.data
ax.pcolormesh(lons, lats, background)
ax.contourf(lons, lats, results_gprof.surface_precip.data, norm=norm, cmap=cmap, levels=levels, extend="both")
ax.coastlines(color="grey")
ax.set_title("(b) GPROF V7", loc="left")
add_ticks(ax, lons=lon_ticks, lats=lat_ticks, left=False)

ax = fig.add_subplot(gs[0, 2], projection=crs)
lons = results_mrms.longitude.data
lats = results_mrms.latitude.data
ax.pcolormesh(lons, lats, background)
ax.contourf(lons, lats, results_gprof_nn_1d.surface_precip.data, norm=norm, cmap=cmap, levels=levels, extend="both")
ax.coastlines(color="grey")
ax.set_title("(c) GPROF-NN 1D", loc="left")
add_ticks(ax, lons=lon_ticks, lats=lat_ticks, left=False)

ax = fig.add_subplot(gs[0, 3], projection=crs)
lons = results_mrms.longitude.data
lats = results_mrms.latitude.data
ax.pcolormesh(lons, lats, background)
m = ax.contourf(lons, lats, results_gprof_nn_3d.surface_precip.data, norm=norm, cmap=cmap, levels=levels, extend="both")
ax.coastlines(color="grey")
ax.set_title("(d) GPROF-NN 3D", loc="left")
add_ticks(ax, lons=lon_ticks, lats=lat_ticks, left=False)

ax = fig.add_subplot(gs[0, -1])
plt.colorbar(m, cax=ax, label="Surface precipitation [mm h$^{-1}$]")
fig.savefig("case_study.png")

In [ ]:
import cartopy.crs as ccrs
from gprof_nn.plotting import add_ticks, scale_bar, get_blue_marble
from matplotlib.gridspec import GridSpec
from matplotlib.colors import Normalize, LogNorm
from pyresample.geometry import SwathDefinition

fig = plt.figure(figsize=(26, 6))
gs = GridSpec(1, 5, width_ratios=[1.0, 1.0, 1.0, 1.0, 0.075])
crs = ccrs.PlateCarree()

levels = np.logspace(-1, 2, 11)
norm = LogNorm(1e-1, 1e1)
levels = np.linspace(0, 20, 11)
norm = Normalize(0, 20)

cmap = "flare"

lon_ticks = [-96, -93, -90]
lat_ticks = [28, 31, 34]

ax = fig.add_subplot(gs[0, 0], projection=crs)
lons = results_mrms.longitude.data
lats = results_mrms.latitude.data
ax.pcolormesh(lons, lats, background)
sp_mrms = results_mrms.surface_precip.data
sp_mrms[sp_mrms < 0] = np.nan
sp_mrms = np.maximum(sp_mrms, 1e-3)
ax.pcolormesh(lons, lats, background)
ax.contourf(lons, lats, sp_mrms, norm=norm, cmap=cmap, levels=levels, extend="both")
ax.coastlines(color="grey")
ax.set_title("(a) MRMS", loc="left")
scale_bar(ax, 400e3, textcolor="w", border_color="w")
add_ticks(ax, lons=lon_ticks, lats=lat_ticks)

ax = fig.add_subplot(gs[0, 1], projection=crs)
lons = results_mrms.longitude.data
lats = results_mrms.latitude.data
ax.pcolormesh(lons, lats, background)
ax.contourf(lons, lats, results_gprof.surface_precip.data, norm=norm, cmap=cmap, levels=levels, extend="both")
ax.coastlines(color="grey")
ax.set_title("(b) GPROF V7", loc="left")
add_ticks(ax, lons=lon_ticks, lats=lat_ticks, left=False)

ax = fig.add_subplot(gs[0, 2], projection=crs)
lons = results_mrms.longitude.data
lats = results_mrms.latitude.data
ax.pcolormesh(lons, lats, background)
ax.contourf(lons, lats, results_gprof_nn_1d.surface_precip.data, norm=norm, cmap=cmap, levels=levels, extend="both")
ax.coastlines(color="grey")
ax.set_title("(c) GPROF-NN 1D", loc="left")
add_ticks(ax, lons=lon_ticks, lats=lat_ticks, left=False)

ax = fig.add_subplot(gs[0, 3], projection=crs)
lons = results_mrms.longitude.data
lats = results_mrms.latitude.data
ax.pcolormesh(lons, lats, background)
m = ax.contourf(lons, lats, results_gprof_nn_3d.surface_precip.data, norm=norm, cmap=cmap, levels=levels, extend="both")
ax.coastlines(color="grey")
ax.set_title("(d) GPROF-NN 3D", loc="left")
add_ticks(ax, lons=lon_ticks, lats=lat_ticks, left=False)

ax = fig.add_subplot(gs[0, -1])
plt.colorbar(m, cax=ax, label="Surface precipitation [mm h$^{-1}$]")
fig.savefig("case_study.png")

## Metrics

### GPROF

In [ ]:
gprof_evaluator.evaluate()
gprof_results = gprof_evaluator.get_results()

In [ ]:
gprof_results_land = gprof_evaluator.get_results_land()
gprof_results_land["Algorithm"] = "GPROF V7"

## GPROF-NN 1D

In [ ]:
gprof_nn_1d_evaluator.evaluate()
gprof_nn_1d_results = gprof_nn_1d_evaluator.get_results()

In [ ]:
gprof_nn_1d_results_land = gprof_nn_1d_evaluator.get_results_land()
gprof_nn_1d_results_land["Algorithm"] = "GPROF-NN 1D"

## GPROF-NN 3D

In [ ]:
gprof_nn_3d_evaluator.evaluate()
gprof_nn_3d_results = gprof_nn_3d_evaluator.get_results()

In [ ]:
gprof_nn_3d_results_land = gprof_nn_3d_evaluator.get_results_land()
gprof_nn_3d_results_land["Algorithm"] = "GPROF-NN 3D"

In [ ]:
gprof_nn_3d_results["Algorithm"] = "GPROF-NN 3D"

In [ ]:
gprof_results.Algorithm

In [ ]:
import pandas as pd
results = xr.concat([gprof_results_land, gprof_nn_2d_results_land, gprof_nn_3d_results_land], "algorithm")
#results = results.drop_vars(("spectral_coherence", "scales")).to_dataframe()

In [ ]:
import seaborn as sns
from matplotlib.gridspec import GridSpec
sns.reset_orig()

fig = plt.figure(figsize=(24, 6))
gs = GridSpec(1, 4, wspace=0.4)

palette = ["C0", "C1", "C2"]

ax = fig.add_subplot(gs[0, 0])
sns.barplot(x="Algorithm", y="bias", data=results, ax=ax, palette=palette, legend=False)
ax.set_ylim(-20, 20)
ax.set_ylabel("Bias [%]")
ax.set_xlabel("Algorithm")
for l in ax.get_xticklabels():
    l.set_rotation(45)
    l.set_horizontalalignment("right")
ax.set_title("(a) Bias", loc="left")
    
ax = fig.add_subplot(gs[0, 1])
sns.barplot(x="Algorithm", y="mae", data=results, ax=ax, palette=palette, legend=False)
ax.set_ylabel("MAE [mm h$^{-1}$]")
ax.set_xlabel("Algorithm")
for l in ax.get_xticklabels():
    l.set_rotation(45)
    l.set_horizontalalignment("right")
ax.set_title("(b) MAE", loc="left")
    
ax = fig.add_subplot(gs[0, 2])
sns.barplot(x="Algorithm", y="mse", data=results, ax=ax, palette=palette, legend=False)
ax.set_ylabel("MSE [(mm h$^{-1}$])$^{2}$")
ax.set_xlabel("Algorithm")
for l in ax.get_xticklabels():
    l.set_rotation(45)
    l.set_horizontalalignment("right")
ax.set_title("(c) MSE", loc="left")

ax = fig.add_subplot(gs[0, 3])
hndls = sns.barplot(x="Algorithm", y="correlation_coef", data=results, ax=ax, palette=palette, legend=False)
ax.set_ylabel("Correlation coef. ")
ax.set_xlabel("Algorithm")
ax.set_ylim(0, 1)
for l in ax.get_xticklabels():
    l.set_rotation(45)
    l.set_horizontalalignment("right")
ax.set_title("(d) Correlation coef.", loc="left")

In [ ]:
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt


fig = plt.figure(figsize=(10, 6))
gs = GridSpec(1, 2, width_ratios=[1.0, 0.2])

results_gprof = gprof_evaluator.get_results()
results_gprof_nn_1d = gprof_nn_1d_evaluator.get_results()
results_gprof_nn_3d = gprof_nn_3d_evaluator.get_results()


ax = fig.add_subplot(gs[0, 0])
x = results_gprof.scales
handles = ax.plot(x, results_gprof.spectral_coherence, label="GPROF V7")
handles += ax.plot(x, results_gprof_nn_1d.spectral_coherence, label="GPROF-NN 1D")
handles += ax.plot(x, results_gprof_nn_3d.spectral_coherence, label="GPROF-NN 3D")
ax.axhline(y=np.sqrt(0.5), ls="--", c="k")
ax.set_ylim(0, 1)
ax.set_ylabel("Spectral coherence")
ax.set_xlabel("Spatial scale [$^\degree$]")
ax.set_title("Spectral Coherence", loc="center")

ax = fig.add_subplot(gs[0, 1])
ax.set_axis_off()
ax.legend(handles=handles, loc="center")

In [ ]:
results_gprof_nn_1d = evaluator.get_results()

results_gprof_nn_1d

In [ ]:
gprof_nn_3d_nrt_path = Path("/data/gprof_v8/results/mrms/gmi/gprof_nn_3d_nrt/")
gprof_nn_3d_na_path = Path("/data/gprof_v8/results/mrms/gmi/gprof_nn_3d_na/")

## Case study

In [ ]:
reference_data

In [ ]:
atms_observations

In [ ]:
from pansat.utils import resample_data

colloc_ind = 200
colloc = collocations[colloc_ind]

atms_observations = xr.load_dataset(colloc, group="input_data")
lons, lats = np.meshgrid(atms_observations.longitude.data, atms_observations.latitude.data)
area = SwathDefinition(lats=lats, lons=lons)

reference_data = xr.load_dataset(colloc, group="reference_data")
gprof_nn_3d_data = resample_data(load_retrieval_results(gprof_nn_3d_path, colloc), area, radius_of_influence=60e3)
gprof_data = resample_data(load_gprof_results(colloc), area, radius_of_influence=60e3)

In [ ]:
import cartopy.crs as ccrs
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LogNorm

norm = LogNorm(1e-1, 1e2)

fig = plt.figure(figsize=(22, 5))
gs = GridSpec(1, 4, width_ratios=[1.0, 1.0, 1.0, 0.1])
crs = ccrs.PlateCarree()

sp_mrms = reference_data.surface_precip.data
sp_gprof = gprof_data.surface_precipitation.data
sp_gprof_nn_3d = gprof_nn_3d_data.surface_precip.data
valid = (sp_mrms >= 0.0) * (sp_gprof >= 0.0) * (sp_gprof_nn_3d >= 0.0)
corr_gprof = np.corrcoef(sp_mrms[valid], sp_gprof[valid])[0, 1]
corr_gprof_nn_3d = np.corrcoef(sp_mrms[valid], sp_gprof_nn_3d[valid])[0, 1]

ax = fig.add_subplot(gs[0, 0], projection=crs)
lons = reference_data.longitude.data
lats = reference_data.latitude.data
sp_mrms = np.maximum(reference_data.surface_precip.data, 1e-3)
ax.pcolormesh(lons, lats, sp_mrms, norm=norm)
ax.coastlines(color="grey")
ax.set_title("(a) MRMS", loc="left")

ax = fig.add_subplot(gs[0, 1], projection=crs)
lons = gprof_data.longitude.data
lats = gprof_data.latitude.data
sp = gprof_data.surface_precipitation.data
ax.set_title(f"(b) GPROF ({corr_gprof:.2})", loc="left")
#ax.set_title(f"(b) GPROF V7", loc="left")

ax.pcolormesh(lons, lats, sp, norm=norm)
ax.coastlines(color="grey")

ax = fig.add_subplot(gs[0, 2], projection=crs)
lons = gprof_nn_3d_data.longitude.data
lats = gprof_nn_3d_data.latitude.data
sp = gprof_nn_3d_data.surface_precip.data
m = ax.pcolormesh(lons, lats, sp, norm=norm)
ax.coastlines(color="grey")
#ax.set_title(f"(c) GPROF-NN 3D", loc="left")
ax.set_title(f"(b) GPROF-NN 3D ({corr_gprof_nn_3d:.2})", loc="left")

cax = fig.add_subplot(gs[0, -1])
plt.colorbar(m, cax=cax, label="Surface precip [mm h$^{-1}$]")

## Evaluation

In [ ]:
from ipwgml.metrics import Bias, MSE, MAE, CorrelationCoef, SpectralCoherence

In [ ]:
metrics_gprof = [Bias(), MSE(), MAE(), CorrelationCeof(), Sp

In [ ]:
import cartopy.crs as ccrs
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LogNorm

norm = LogNorm(1e-1, 1e2)

fig = plt.figure(figsize=(22, 5))
gs = GridSpec(1, 4, width_ratios=[1.0, 1.0, 1.0, 0.1])
crs = ccrs.PlateCarree()

sp_mrms = reference_data.surface_precip.data
sp_gprof = gprof_data.surface_precipitation.data
sp_gprof_nn_3d = gprof_nn_3d_data.surface_precip.data
valid = (sp_mrms >= 0.0) * (sp_gprof >= 0.0) * (sp_gprof_nn_3d >= 0.0)
corr_gprof = np.corrcoef(sp_mrms[valid], sp_gprof[valid])[0, 1]
corr_gprof_nn_3d = np.corrcoef(sp_mrms[valid], sp_gprof_nn_3d[valid])[0, 1]

ax = fig.add_subplot(gs[0, 0], projection=crs)
lons = reference_data.longitude.data
lats = reference_data.latitude.data
sp_mrms = np.maximum(reference_data.surface_precip.data, 1e-3)
ax.pcolormesh(lons, lats, sp_mrms, norm=norm)
ax.coastlines(color="grey")
ax.set_title("(a) MRMS", loc="left")

ax = fig.add_subplot(gs[0, 1], projection=crs)
lons = gprof_data.longitude.data
lats = gprof_data.latitude.data
sp = gprof_data.surface_precipitation.data
ax.set_title(f"(b) GPROF ({corr_gprof:.2})", loc="left")
#ax.set_title(f"(b) GPROF V7", loc="left")

ax.pcolormesh(lons, lats, sp, norm=norm)
ax.coastlines(color="grey")

ax = fig.add_subplot(gs[0, 2], projection=crs)
lons = gprof_nn_3d_data.longitude.data
lats = gprof_nn_3d_data.latitude.data
sp = gprof_nn_3d_data.surface_precip.data
m = ax.pcolormesh(lons, lats, sp, norm=norm)
ax.coastlines(color="grey")
#ax.set_title(f"(c) GPROF-NN 3D", loc="left")
ax.set_title(f"(b) GPROF-NN 3D ({corr_gprof_nn_3d:.2})", loc="left")

cax = fig.add_subplot(gs[0, -1])
plt.colorbar(m, cax=cax, label="Surface precip [mm h$^{-1}$]")

In [ ]:
import cartopy.crs as ccrs
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LogNorm

norm = LogNorm(1e-1, 1e2)

fig = plt.figure(figsize=(22, 5))
gs = GridSpec(1, 4, width_ratios=[1.0, 1.0, 1.0, 0.1])
crs = ccrs.PlateCarree()

sp_mrms = reference_data.surface_precip.data
sp_gprof = gprof_data.surface_precipitation.data
sp_gprof_nn_3d = gprof_nn_3d_data.surface_precip.data
valid = (sp_mrms >= 0.0) * (sp_gprof >= 0.0) * (sp_gprof_nn_3d >= 0.0)
corr_gprof = np.corrcoef(sp_mrms[valid], sp_gprof[valid])[0, 1]
corr_gprof_nn_3d = np.corrcoef(sp_mrms[valid], sp_gprof_nn_3d[valid])[0, 1]

ax = fig.add_subplot(gs[0, 0], projection=crs)
lons = reference_data.longitude.data
lats = reference_data.latitude.data
sp_mrms = np.maximum(reference_data.surface_precip.data, 1e-3)
ax.pcolormesh(lons, lats, sp_mrms, norm=norm)
ax.coastlines(color="grey")
ax.set_title("(a) MRMS", loc="left")

ax = fig.add_subplot(gs[0, 1], projection=crs)
lons = gprof_data.longitude.data
lats = gprof_data.latitude.data
sp = gprof_data.surface_precipitation.data
ax.set_title(f"(b) GPROF ({corr_gprof:.2})", loc="left")
#ax.set_title(f"(b) GPROF V7", loc="left")

ax.pcolormesh(lons, lats, sp, norm=norm)
ax.coastlines(color="grey")

ax = fig.add_subplot(gs[0, 2], projection=crs)
lons = gprof_nn_3d_data.longitude.data
lats = gprof_nn_3d_data.latitude.data
sp = gprof_nn_3d_data.surface_precip.data
m = ax.pcolormesh(lons, lats, sp, norm=norm)
ax.coastlines(color="grey")
#ax.set_title(f"(c) GPROF-NN 3D", loc="left")
ax.set_title(f"(b) GPROF-NN 3D ({corr_gprof_nn_3d:.2})", loc="left")

cax = fig.add_subplot(gs[0, -1])
plt.colorbar(m, cax=cax, label="Surface precip [mm h$^{-1}$]")

In [ ]:
import cartopy.crs as ccrs
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LogNorm

norm = LogNorm(1e-1, 1e2)

fig = plt.figure(figsize=(22, 5))
gs = GridSpec(1, 4, width_ratios=[1.0, 1.0, 1.0, 0.1])
crs = ccrs.PlateCarree()

sp_mrms = reference_data.surface_precip.data
sp_gprof = gprof_data.surface_precipitation.data
sp_gprof_nn_3d = gprof_nn_3d_data.surface_precip.data
valid = (sp_mrms >= 0.0) * (sp_gprof >= 0.0) * (sp_gprof_nn_3d >= 0.0)
corr_gprof = np.corrcoef(sp_mrms[valid], sp_gprof[valid])[0, 1]
corr_gprof_nn_3d = np.corrcoef(sp_mrms[valid], sp_gprof_nn_3d[valid])[0, 1]

ax = fig.add_subplot(gs[0, 0], projection=crs)
lons = reference_data.longitude.data
lats = reference_data.latitude.data
sp_mrms = np.maximum(reference_data.surface_precip.data, 1e-3)
ax.pcolormesh(lons, lats, sp_mrms, norm=norm)
ax.coastlines(color="grey")
ax.set_title("(a) MRMS", loc="left")

ax = fig.add_subplot(gs[0, 1], projection=crs)
lons = gprof_data.longitude.data
lats = gprof_data.latitude.data
sp = gprof_data.surface_precipitation.data
ax.set_title(f"(b) GPROF ({corr_gprof:.2})", loc="left")
#ax.set_title(f"(b) GPROF V7", loc="left")

ax.pcolormesh(lons, lats, sp, norm=norm)
ax.coastlines(color="grey")

ax = fig.add_subplot(gs[0, 2], projection=crs)
lons = gprof_nn_3d_data.longitude.data
lats = gprof_nn_3d_data.latitude.data
sp = gprof_nn_3d_data.surface_precip.data
m = ax.pcolormesh(lons, lats, sp, norm=norm)
ax.coastlines(color="grey")
#ax.set_title(f"(c) GPROF-NN 3D", loc="left")
ax.set_title(f"(b) GPROF-NN 3D ({corr_gprof_nn_3d:.2})", loc="left")

cax = fig.add_subplot(gs[0, -1])
plt.colorbar(m, cax=cax, label="Surface precip [mm h$^{-1}$]")

In [ ]:
import cartopy.crs as ccrs
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LogNorm

norm = LogNorm(1e-1, 1e2)

fig = plt.figure(figsize=(22, 5))
gs = GridSpec(1, 4, width_ratios=[1.0, 1.0, 1.0, 0.1])
crs = ccrs.PlateCarree()

sp_mrms = reference_data.surface_precip.data
sp_gprof = gprof_data.surface_precipitation.data
sp_gprof_nn_3d = gprof_nn_3d_data.surface_precip.data
valid = (sp_mrms >= 0.0) * (sp_gprof >= 0.0) * (sp_gprof_nn_3d >= 0.0)
corr_gprof = np.corrcoef(sp_mrms[valid], sp_gprof[valid])[0, 1]
corr_gprof_nn_3d = np.corrcoef(sp_mrms[valid], sp_gprof_nn_3d[valid])[0, 1]

ax = fig.add_subplot(gs[0, 0], projection=crs)
lons = reference_data.longitude.data
lats = reference_data.latitude.data
sp_mrms = np.maximum(reference_data.surface_precip.data, 1e-3)
ax.pcolormesh(lons, lats, sp_mrms, norm=norm)
ax.coastlines(color="grey")
ax.set_title("(a) MRMS", loc="left")

ax = fig.add_subplot(gs[0, 1], projection=crs)
lons = gprof_data.longitude.data
lats = gprof_data.latitude.data
sp = gprof_data.surface_precipitation.data
ax.set_title(f"(b) GPROF ({corr_gprof:.2})", loc="left")
#ax.set_title(f"(b) GPROF V7", loc="left")

ax.pcolormesh(lons, lats, sp, norm=norm)
ax.coastlines(color="grey")

ax = fig.add_subplot(gs[0, 2], projection=crs)
lons = gprof_nn_3d_data.longitude.data
lats = gprof_nn_3d_data.latitude.data
sp = gprof_nn_3d_data.surface_precip.data
m = ax.pcolormesh(lons, lats, sp, norm=norm)
ax.coastlines(color="grey")
#ax.set_title(f"(c) GPROF-NN 3D", loc="left")
ax.set_title(f"(b) GPROF-NN 3D ({corr_gprof_nn_3d:.2})", loc="left")

cax = fig.add_subplot(gs[0, -1])
plt.colorbar(m, cax=cax, label="Surface precip [mm h$^{-1}$]")


# Evaluate multiple scenes

In [ ]:
experiments = {
    "GPROF-NN 3D (CLI)": gprof_nn_3d_path,
    #"GPROF-NN 3D (NRT)": gprof_nn_3d_nrt_path,
    #"GPROF-NN 3D (No Anc.)": gprof_nn_3d_na_path,
    #"GPROF-NN 3D (ATMS, CLI)": gprof_nn_3d_atms_path,
    #"GPROF-NN 3D (ATMS, No Anc.)": gprof_nn_3d_path,
}

In [ ]:
from tqdm import tqdm
sp_ref = []
sp_gprof = []
sp_gprof_nn = {}
eia = []
surface_type = []
rqi = []

scene_ind = 0
for colloc in tqdm(np.random.permutation(collocations)[:200]):

    try:
        atms_observations = xr.load_dataset(colloc, group="input_data")
        lons, lats = np.meshgrid(atms_observations.longitude.data, atms_observations.latitude.data)
        area = SwathDefinition(lats=lats, lons=lons)
    
        reference_data = xr.load_dataset(colloc, group="reference_data")
        input_data = xr.load_dataset(colloc, group="input_data")
        gprof_nn_data = {
            name: resample_data( load_retrieval_results(path, colloc), area, radius_of_influence=60e3)
            for name, path in experiments.items()
        }
        gprof_data = resample_data(load_gprof_results(colloc), area, radius_of_influence=60e3)

        sp_ref_c = reference_data.surface_precip.data
        rqi_c = reference_data.radar_quality_index.data
        sp_gprof_c = gprof_data.surface_precipitation.data
        sp_gprof_nn_c = {name: data.surface_precip.data for name, data in gprof_nn_data.items()}
        surface_type_c = input_data.surface_type.data
        eia_c = input_data.earth_incidence_angle.data

        valid = (
            (-10 <= sp_ref_c) *
            (-10 <= sp_gprof_c)
        )
        for sp in sp_gprof_nn_c.values():
            valid *= (-10 <= sp)
        
        sp_ref.append(sp_ref_c[valid])
        sp_gprof.append(sp_gprof_c[valid])
        for name, sp in sp_gprof_nn_c.items():
            sp_gprof_nn.setdefault(name, []).append(sp[valid])
        surface_type.append(surface_type_c[valid])
        eia.append(eia_c[valid])
        rqi.append(rqi_c[valid])

        
        if sp.max() > 1_000:
            print(colloc)
            
    except Exception as exc:
        print(exc)
        pass


In [ ]:
results = xr.Dataset({
    "sp_ref":  (("samples"), np.concatenate(sp_ref)),
    "sp_gprof": (("samples"), np.concatenate(sp_gprof)),
    "surface_type": ("samples", np.concatenate(surface_type)),
    "eia": ("samples", np.concatenate(eia)[:, 0]),
    "rqi": ("samples", np.concatenate(rqi)),
})
for name, sp in sp_gprof_nn.items():
    results[f"sp_{name}"] = (("samples",), np.concatenate(sp))

In [ ]:
results["sp_GPROF-NN 3D (CLI)"].data

In [ ]:
import pandas as pd

def calculate_error_stats(results):
    sp_ref = results.sp_ref.data
    sp_gprof = results.sp_gprof.data
    sp_gprof_nn = {name: results[f"sp_{name}"].data for name in experiments.keys()}
    
    bias_gprof = 100.0 * (sp_gprof - sp_ref).mean() / sp_ref.mean()
    biases = [bias_gprof]
    biases += [100.0 * (sp_nn - sp_ref).mean() / sp_ref.mean() for sp_nn in sp_gprof_nn.values()]
    
    corr_gprof = np.corrcoef(sp_ref, sp_gprof)[0, 1]
    corrs = [corr_gprof]
    corrs += [np.corrcoef(sp_ref, sp_nn)[0, 1] for sp_nn in sp_gprof_nn.values()]
    
    mse_gprof = np.mean((sp_ref - sp_gprof)**2)
    mses = [mse_gprof]
    mses += [np.mean((sp_ref - sp_nn)**2) for sp_nn in sp_gprof_nn.values()]
    
    mae_gprof = np.mean(np.abs(sp_ref - sp_gprof))
    maes = [mae_gprof]
    maes += [np.mean(np.abs(sp_ref - sp_nn)) for sp_nn in sp_gprof_nn.values()]
    
    return pd.DataFrame({
        "Algorithm": ["GPROF"] + list(sp_gprof_nn.keys()),
        "Bias": biases, 
        "MSE": mses,
        "MAE": maes,
        "Correlation": corrs
    })
    

In [ ]:
results_veg = results[{"samples": (2 < results.surface_type) * (results.surface_type < 8) * (results.rqi >= 0.8)}]

In [ ]:
stats_veg = calculate_error_stats(results_veg)

In [ ]:
stats_veg

In [ ]:
stats_veg

In [ ]:
stats_veg

In [ ]:
results_ocean = results[{"samples": (results.surface_type == 1) * (results.rqi >= 0.8)}]

In [ ]:
results_ocean

In [ ]:
stats_ocean = calculate_error_stats(results_ocean)

In [ ]:
stats_ocean

In [ ]:
import seaborn as sns
sns.reset_orig()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 8))

# Bias plot
sns.barplot(x='Algorithm', y='Bias', data=stats_veg, ax=axes[0])
axes[0].set_title("Bias")
for l in axes[0].get_xticklabels():
    l.set_rotation(45)

# MSE and MAE plot
sns.barplot(x='Algorithm', y='MSE', data=stats_veg, ax=axes[1])
axes[1].set_title("MSE")
for l in axes[1].get_xticklabels():
    l.set_rotation(45)
    
sns.barplot(x='Algorithm', y='MAE', data=stats_veg, ax=axes[2])
axes[2].set_title("MAE")
for l in axes[2].get_xticklabels():
    l.set_rotation(45)

# Correlation plot
sns.barplot(x='Algorithm', y='Correlation', data=stats_veg, ax=axes[3])
axes[3].set_title("Correlation")
for l in axes[3].get_xticklabels():
    l.set_rotation(45)

# Adjust layout and show the plots
plt.tight_layout()
plt.show()

In [ ]:
results_coast = results[{"samples": (11 < results.surface_type) * (results.surface_type < 16) * (results.rqi >= 0.8)}]

In [ ]:
calculate_error_stats(results_coast)

In [ ]:
results_snow = results[{"samples": (8 <= results.surface_type) * (results.surface_type <= 12) * results.rqi > 0.8}]

In [ ]:
calculate_error_stats(results_snow)

In [ ]:
results = results[{"samples": results.sp_gprof_nn_sf.data < 1_000}]
results.to_netcdf("results_gprofnn.nc")

In [ ]:
results = xr.load_dataset("results_gprofnn.nc")

In [ ]:
results_ocean = results[{"samples": results.surface_type.data == 1}]
calculate_error_stats(results_ocean)

In [ ]:
results_ocean = results[{"samples": results.surface_type.data == 1}]
calculate_error_stats(results_ocean)

In [ ]:
results_veg = results[{"samples": (2 < results.surface_type.data) * (results.surface_type.data < 8)}]
calculate_error_stats(results_veg)

In [ ]:
results_veg = results[{"samples": (2 < results.surface_type.data) * (results.surface_type.data < 8)}]
calculate_error_stats(results_veg)

In [ ]:
results_veg = results[{"samples": (11 < results.surface_type.data) * (results.surface_type.data < 16)}]
calculate_error_stats(results_veg)

In [ ]:
results_veg = results[{"samples": (11 < results.surface_type.data) * (results.surface_type.data < 16)}]
calculate_error_stats(results_veg)

In [ ]:
np.mean(results_ocean.sp_gprof.data - results_ocean.sp_ref.data) / results_ocean.sp_ref.data.mean()

In [ ]:
np.corrcoef(results_ocean.sp_ref.data, results_ocean.sp_gprof.data)[0, 1]

In [ ]:
np.corrcoef(results_ocean.sp_ref.data, results_ocean.sp_gprof_nn_sf.data)[0, 1]

In [ ]:
results_ocean.sp_gprof_nn_sf.data.mean()

In [ ]:
results_ocean.sp_gprof.data.mean()

In [ ]:
results_ocean.sp_ref.data.mean()

In [ ]:
((results_ocean.sp_gprof.data - results_ocean.sp_ref.data)**2).mean()

In [ ]:
((results_ocean.sp_gprof_nn_sf.data - results_ocean.sp_ref.data)**2).mean()

In [ ]:
from scipy.stats import binned_statistic
results.eia.data.max()
bins = np.linspace(0, 50, 6)
va_mean_sf = binned_statistic(results_ocean.eia.data, results_ocean.sp_gprof_nn_sf.data)[0]
va_mean_gprof = binned_statistic(results_ocean.eia.data, results_ocean.sp_gprof.data)[0]
va_mean_ref = binned_statistic(results_ocean.eia.data, results_ocean.sp_ref.data)[0]


In [ ]:
plt.plot(va_mean_sf / va_mean_ref, label="SF")
#plt.plot(va_mean_gprof, label="GPROF V7")
#plt.plot(va_mean_ref, label="REF")
plt.legend()

In [ ]:
plt.plot(va_mean_sf, label="SF")
plt.plot(va_mean_gprof, label="GPROF V7")
plt.plot(va_mean_ref, label="REF")
plt.legend()

In [ ]:
stats = xr.load_dataset("/gdata1/simon/gprof_v8/models/atms/gprof_nn_1d_sf/stats/input/viewing_angles.nc")


In [ ]:
stats["max"]

In [ ]:
plt.plot(stats["counts"][0])

In [ ]:
np.mean(results_ocean.sp_gprof_nn_sf.data ) / results_ocean.sp_ref.data.mean()

In [ ]:
from matplotlib.gridspec import GridSpec

CHANNELS = [
    "89 GHz",
    "164 GHz",
    "183 +/- 1 GHz",
    "183 +/- 3 GHz",
    "183 +/- 7 GHz",
]

def make_scater_plots(results):
    fig = plt.figure(figsize=(20, 25))
    gs = GridSpec(5, 5, width_ratios=[0.3, 1.0, 1.0, 1.0, 1.0])
    
    
    for chan in range(5):
        
        ax = fig.add_subplot(gs[chan, 0])
        ax.set_axis_off()
        ax.text(0, 0, CHANNELS[chan], rotation=90, ha="center", va="center")
        ax.set_ylim(-2, 2)
        
        tbs_act = results["tbs_actual"].data[..., chan]
        tbs_sim = results["tbs_sim"].data[..., chan]
        tbs_bias = results["tbs_sim_bias"].data[..., chan]
        tbs_sf = results["tbs_sf"].data[..., chan]
        eia = results["eia"].data
        
        #
        # Simulated TBS
        #
        
        ax = fig.add_subplot(gs[chan, 1])
        if chan == 0:
            ax.set_title("Simulated", loc="center")
        bins = np.linspace(tbs_act.min(), tbs_act.max())
        tbs = tbs_sim
        dens = np.histogram2d(tbs_act, tbs, bins=bins)[0]
        dens /= dens.sum(1, keepdims=True)
        x = 0.5 * (bins[1:] + bins[:-1])
        ax.pcolormesh(x, x, dens.T)
        ax.plot(x, x, ls="--", c="grey")
        ax.set_aspect(1.0)
        
        bias = 100.0 * (tbs - tbs_act).mean() / np.mean(tbs_act)
        rmse = np.sqrt(np.mean((tbs - tbs_act) ** 2))
        corr = np.corrcoef(tbs_act, tbs)[0, 1]
        ax.text(0.1, 0.7, f"Bias:  {bias:.2f} %\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}", transform=ax.transAxes, color="grey")

        ax.set_ylabel("Simulated $T_b$ [K]")
        ax.set_xlabel("Actual $T_b$ [K]")
        ax.grid(False)
        
        #
        # Bias-corrected, simulated TBs
        #
        
        ax = fig.add_subplot(gs[chan, 2])
        if chan == 0:
            ax.set_title("Simulated - Bias", loc="center")
        tbs = tbs_sim - tbs_bias
        dens = np.histogram2d(tbs_act, tbs, bins=bins)[0]
        dens /= dens.sum(1, keepdims=True)
        x = 0.5 * (bins[1:] + bins[:-1])
        ax.pcolormesh(x, x, dens.T)
        ax.plot(x, x, ls="--", c="grey")
        ax.set_aspect(1.0)
        
        bias = 100.0 * (tbs - tbs_act).mean() / np.mean(tbs_act)
        rmse = np.sqrt(np.mean((tbs - tbs_act) ** 2))
        corr = np.corrcoef(tbs_act, tbs)[0, 1]
        ax.text(0.1, 0.7, f"Bias:  {bias:.2f} %\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}", transform=ax.transAxes, color="grey")
        
        ax.set_yticklabels([])
        ax.set_xlabel("Actual $T_b$ [K]")
        ax.grid(False)
        
        #
        # EIA adapted bias correction
        #
        
        ax = fig.add_subplot(gs[chan, 3])
        if chan == 0:
            ax.set_title(r"Simulated - $\frac{\cos(\theta_{\text{GMI}})}{\cos(\theta)}$ Bias", loc="center")
        tbs = tbs_sim - np.cos(np.deg2rad(48.0)) / np.cos(np.deg2rad(eia)) * tbs_bias
        dens = np.histogram2d(tbs_act, tbs, bins=bins)[0]
        dens /= dens.sum(1, keepdims=True)
        x = 0.5 * (bins[1:] + bins[:-1])
        ax.pcolormesh(x, x, dens.T)
        ax.plot(x, x, ls="--", c="grey")
        ax.set_aspect(1.0)
        
        bias = 100.0 * (tbs - tbs_act).mean() / np.mean(tbs_act)
        rmse = np.sqrt(np.mean((tbs - tbs_act) ** 2))
        corr = np.corrcoef(tbs_act, tbs)[0, 1]
        ax.text(0.1, 0.7, f"Bias:  {bias:.2f} %\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}", transform=ax.transAxes, color="grey")
        
        ax.set_yticklabels([])
        ax.set_xlabel("Actual $T_b$ [K]")
        ax.grid(False)
        
        #
        # Satformer results
        #
        
        ax = fig.add_subplot(gs[chan, 4])
        if chan == 0:
            ax.set_title(r"Satformer", loc="center")
        
        tbs = tbs_sf
        dens = np.histogram2d(tbs_act, tbs, bins=bins)[0]
        dens /= dens.sum(1, keepdims=True)
        x = 0.5 * (bins[1:] + bins[:-1])
        ax.pcolormesh(x, x, dens.T)
        ax.plot(x, x, ls="--", c="grey")
        ax.set_aspect(1.0)
                                 
        bias = 100.0 * (tbs - tbs_act).mean() / np.mean(tbs_act)
        rmse = np.sqrt(np.mean((tbs - tbs_act) ** 2))
        corr = np.corrcoef(tbs_act, tbs)[0, 1]
        #ax.text(0.1 * x[0], 0.9 * x[-1], f"Bias:  {bias:.2f}\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}")
        ax.text(0.1, 0.7, f"Bias:  {bias:.2f} %\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}", transform=ax.transAxes, color="grey")
        
        ax.set_yticklabels([])
        ax.set_xlabel("Actual $T_b$ [K]")
        ax.grid(False)

    return fig, ax

In [ ]:
100

In [ ]:
fig, ax = make_scater_plots(results)

In [ ]:
fig, ax = make_scater_plots(results)

In [ ]:
fig, ax = make_scater_plots(results)

In [ ]:
fig, ax = make_scater_plots(results)

In [ ]:
results_h = results[{"samples": np.abs(results.eia) > 40}]
make_scater_plots(results_h)

In [ ]:
results_h = results[{"samples": np.abs(results.eia) > 40}]
make_scater_plots(results_h)

## Test simulations

In [ ]:
time_range = TimeRange("2020-06-03", "2020-06-04")
gmi_recs = l1c_r_gpm_gmi.get(time_range=time_range)
atms_recs = l1c_noaa20_atms.get(time_range=time_range)
gmi_index = Index.index(l1c_r_gpm_gmi, gmi_recs)
atms_index = Index.index(l1c_noaa20_atms, atms_recs)
matches = find_matches(gmi_index, atms_index)
input_loader = InputLoader(matches)

In [ ]:
inpt, fname, aux = input_loader.load_data(1)

In [ ]:
inpt["observations"].shape

In [ ]:
plt.pcolormesh(inpt["observations"][0, 2, 6])

In [ ]:
inpt.keys()

In [ ]:
tile = {name: tensor[..., 350:478, 20:148] for name, tensor in inpt.items()}
for name, tensor in inpt.items():
    if name.endswith("_mask"):
        tile[name] = inpt[name]
tbs_targ = aux["target_observations"].data[..., 64:128, 64:128]

In [ ]:
tile["observations"].shape

In [ ]:
plt.imshow(tile["observations"][0, 2, 0])

In [ ]:
import torch

with torch.no_grad():
    y_pred = model(tile)["output_observations"]
    y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

In [ ]:
from copy import deepcopy

props = tile["output_observation_props"].clone()
props[:, :, 1:] = props[:, :, :1]
props[0, 3, :] = torch.tensor(np.linspace(1.75, 5.2, 9))[..., None, None]
props[0, 4, :] = torch.tensor(np.linspace(416e3, 800e3, 9))[..., None, None]

tile_bw = deepcopy(tile)
tile_bw["output_observation_props"] = props

In [ ]:

with torch.no_grad():
    y_pred = model(tile_bw)["output_observations"]
    y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

## Beam width

In [ ]:
def simulate_beam_width(beam_width):
    
    props = tile["output_observation_props"].clone()[:, :, :1]
    props[0, 3, :] = beam_width
    tile_bw = deepcopy(tile)
    tile_bw["output_observation_props"] = props

    with torch.no_grad():
        y_pred = model(tile_bw)["output_observations"]
        y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

    plt.figure(figsize=(6.5, 5))
    plt.title(f"89 GHz, beam width = {beam_width:.2f} deg.")
    plt.imshow(y_pred[0], vmin=160, vmax=240)
    plt.colorbar(label="$T_b$ [K]")
    plt.grid(False)


In [ ]:
model

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact

interact(simulate_beam_width, beam_width=(1.5, 5.2))


## WV-channel offset

In [ ]:
def simulate_offset(offset):
    
    props = tile["output_observation_props"].clone()[:, :, -1:]
    props[0, 1, :] = offset
    tile_bw = deepcopy(tile)
    tile_bw["output_observation_props"] = props

    with torch.no_grad():
        y_pred = model(tile_bw)["output_observations"]
        y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

    plt.figure(figsize=(6.5, 5))
    plt.title(f"183 +/- {offset:.2f} GHz")
    plt.imshow(y_pred[0], vmin=240, vmax=270)
    plt.colorbar(label="$T_b$ [K]")
    plt.grid(False)

In [ ]:
interact(simulate_offset, offset=(1.0, 7.0))

## EIA

In [ ]:
def simulate_eia(eia):
    
    props = tile["output_observation_props"].clone()[:, :, [-5]]
    props[0, -2, :] += eia
    tile_bw = deepcopy(tile)
    tile_bw["output_observation_props"] = props

    with torch.no_grad():
        y_pred = model(tile_bw)["output_observations"]
        y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

    plt.figure(figsize=(6.5, 5))
    plt.title(rf"183 +/- 7 GHz, $\Delta\theta = ${eia:.2f}")
    plt.imshow(y_pred[0], vmin=220, vmax=280)
    plt.colorbar(label="$T_b$ [K]")
    plt.grid(False)

In [ ]:
interact(simulate_eia, eia=(-20.0, 20.0))

In [ ]:
plt.imshow(props[0, -2, -1])
plt.colorbar()

## Run simul